# Mode-classification calibration on GS2 linear databases (Fusion_PhD-7k70, lqwx, use8)

Calibrates the cutoffs of `general_analysis.mode_classification` on every GS2 Latin-hypercube database with per-species fluxes.
Three indicators per case from the final time: tearing parameter `T` (Hatch 2012), signed frequency `omega` (ion direction > 0, pyrokinetics
convention) and `chi_e/chi_i` (Kotschenreuther 2019). Consensus: a label only if every indicator agrees, otherwise `mixed`.
**Fusion_PhD-lqwx (2026-10-01):** the user's final criteria (KBM: T<0.05, omega>0, 0.25<chi_e/chi_i<4; MTM: T>0.15, omega<0).
**Fusion_PhD-use8 (2026-10-05):** indicators now come from each database's cube (`pyroscan.nc`-style `cube.nc` + `pyroscan.json`) through
`mc.indicators(scan, ds)` (`scan.sample_pyro`, Fusion_PhD-k8w8); no run directory is read and no per-run `Pyro` is built. The D_e/chi_e cut and the
7k70 old->new transition table (which needed it) were dropped by the user and are gone with `de_chi`.

In [ ]:
import os, warnings
from multiprocessing import Pool
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
from general_analysis import mode_classification as mc

warnings.filterwarnings("ignore")
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
load_dotenv(ROOT / "local.env")
data_root = Path(os.environ["GK_DATA_ROOT"])
code_, run_template, project = "GS2", "Runs", "LATIN_HYPERCUBE"
# database label: (case, scan_information)
# database label: (case, scan_information). Every GS2 LATIN_HYPERCUBE database; load failures are reported.
databases = {"NSTX_MTM n1000": ("NSTX_MTM", "beta_q_shat_ky_n1000"), "NSTX_MTM n300": ("NSTX_MTM", "beta_q_shat_ky_n300"),
             "SPR-045": ("SPR-045", "kbm_8d"), "M1": ("M1", "kbm_8d"), "R4": ("R4", "kbm_8d")}
databases |= {f"SPR-008 {s}": ("SPR-008", s) for s in ("aLTe_shat", "aLTe_q", "aLTe_aLn", "aLTe_aLn_shat", "aLTe_aLn_q_n200",
              "aLTe_aLn_q_n500", "aLTe_aLn_shat_q_n200", "aLTe_aLn_shat_q_n500")}
n_spot, seed = 5, 0
plot_dir = ROOT / "Plots" / "mode_classification_lqwx"; plot_dir.mkdir(parents=True, exist_ok=True)
plt.style.use(Path(mc.__file__).parent / "paper.mplstyle")

In [ ]:
rows, cubes = [], {}
for db, (case, scan_name) in databases.items():
    try:
        scan, ds = mc.load_gs2_cube(data_root / code_ / run_template / project / case / scan_name)
        ind = mc.indicators(scan, ds).to_dataframe()
        cubes[db] = ds
        gamma = ds["growth_rate"].squeeze().pint.dequantify().values
        rows.append(ind.assign(db=db, run=ind.sample_name, growth_rate=gamma))
    except Exception as e:
        rows.append(pd.DataFrame([dict(db=db, run="", err=repr(e)[:120])]))
raw = pd.concat(rows, ignore_index=True)

In [ ]:
# Nothing is dropped silently: load failures, then non-positive growth rate, then non-finite indicators.
failed = raw[raw.get("err").notna()] if "err" in raw else raw.iloc[:0]
ok = raw.drop(failed.index)
failed.to_csv(plot_dir / "load_failures.csv", index=False)
print(failed.groupby("db").err.first().to_dict())
print("loaded", raw.groupby("db").size().to_dict(), "| load failures", failed.groupby("db").size().to_dict())
stable = ok[ok.growth_rate <= 0]
ok = ok[ok.growth_rate > 0]
nonfinite = ok[~np.isfinite(ok[["T", "omega", "chi_ratio"]]).all(axis=1)]
df = ok.drop(nonfinite.index).copy()
print("growth_rate <= 0", stable.groupby("db").size().to_dict(), "| non-finite indicator (zero flux)", nonfinite.groupby("db").size().to_dict())
print("kept", df.groupby("db").size().to_dict())

## Sign convention check
pyrokinetics states `mode_frequency < 0` is the electron direction (comments in its CGYRO and GENE readers). Check on the
unambiguous populations: NSTX_MTM cases with tearing parity and huge `chi_e/chi_i` are MTMs and must be electron direction;
SPR-045 + M1 cases with ballooning parity and `chi_e/chi_i` ~ 1 are the KBM population.

In [ ]:
mtm_like = df[(df["T"] > 0.7) & (df.chi_ratio > 30)]
kbm_like = df[(df["T"] < 0.05) & df.chi_ratio.between(0.5, 2) & ~df.db.str.startswith("NSTX")]
print(f"MTM-like (T>0.7, chi_e/chi_i>30): {len(mtm_like)} cases, omega<0 (electron) in {(mtm_like.omega < 0).mean():.3f}")
print(f"KBM-like (T<0.05, chi 0.5-2): {len(kbm_like)} cases, omega>0 (ion) in {(kbm_like.omega > 0).mean():.3f}")

## Indicator gs2_mode_indicator_histograms and pairwise scatters (cutoffs from the module drawn as lines)

In [ ]:
col = dict(zip(df.db.unique(), plt.cm.tab20.colors))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
bins = {"T": np.logspace(-6, 0, 40), "omega": np.linspace(-5, 5, 50), "chi_ratio": np.logspace(-2, 3.5, 40)}
for a, (k, b) in zip(ax, bins.items()):
    for db, g in df.groupby("db"):
        a.hist(np.clip(g[k], b[0], b[-1]), bins=b, histtype="step", label=db, color=col[db])
    a.set_xlabel(k)
    if k in ("T", "chi_ratio"): a.set_xscale("log")
ax[0].axvline(mc.T_BAL, c="k", ls="--"); ax[0].axvline(mc.T_TEAR, c="k", ls=":")
ax[1].axvline(0, c="k", ls="--")
ax[2].axvspan(*mc.CHI_KBM, color="grey", alpha=0.2)
ax[0].legend(fontsize=5)
fig.savefig(plot_dir / "gs2_mode_indicator_histograms.png")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [("T", "chi_ratio"), ("omega", "chi_ratio"), ("omega", "T")]
for a, (x, y) in zip(ax.flat, pairs):
    for db, g in df.groupby("db"):
        a.scatter(g[x], g[y], s=6, alpha=0.5, label=db, c=col[db])
    a.set_xlabel(x); a.set_ylabel(y)
    if x == "T": a.set_xscale("symlog", linthresh=1e-3)
    if y == "chi_ratio": a.set_yscale("symlog", linthresh=0.1); a.axhspan(*mc.CHI_KBM, color="grey", alpha=0.15)
    if y == "T": a.set_yscale("symlog", linthresh=1e-3); a.axhline(mc.T_BAL, c="k", ls="--"); a.axhline(mc.T_TEAR, c="k", ls=":")
    if x == "T": a.axvline(mc.T_BAL, c="k", ls="--"); a.axvline(mc.T_TEAR, c="k", ls=":")
    if x == "omega": a.axvline(0, c="k", ls="--")
ax[0].legend(fontsize=5)
fig.savefig(plot_dir / "gs2_mode_indicator_pairwise_scatter.png")